In [3]:
# 必要なライブラリをインポート
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib # モデルの保存に使用
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error
import os

# --- 1. データの読み込み ---
features_data_path = '../data/processed/features.parquet'
df = pd.read_parquet(features_data_path)

In [10]:
!pip install statsmodels

   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   ---------- ----------------------------- 2.6/9.6 MB 15.1 MB/s eta 0:00:01
   --------------------- ------------------ 5.2/9.6 MB 12.3 MB/s eta 0:00:01
   ----------------------------- ---------- 7.1/9.6 MB 11.2 MB/s eta 0:00:01
   ---------------------------------------  9.4/9.6 MB 11.3 MB/s eta 0:00:01
   ---------------------------------------- 9.6/9.6 MB 10.7 MB/s eta 0:00:00


In [43]:
### ①ライブラリの読み込み ###
import statsmodels.api as sm #統計モデルパッケージを読み込み
from sklearn import preprocessing 

### ②説明変数・目的変数のセット ###
# 説明変数のセット
X = df[['最寄駅：距離（分）', '面積（㎡）',  '建ぺい率（％）',
       '容積率（％）','取引時点での築年数', '取引の事情等_その他事情有り',
       '取引の事情等_他の権利・負担付き', '取引の事情等_他の権利・負担付き、調停・競売等', '取引の事情等_瑕疵有りの可能性',
       '取引の事情等_調停・競売等', '取引の事情等_調停・競売等、瑕疵有りの可能性', '取引の事情等_関係者間取引',
       '取引の事情等_関係者間取引、瑕疵有りの可能性', '取引の事情等_関係者間取引、調停・競売等', 
       '改装_改装済',  '間取り_grouped_オープンフロア',
       '間取り_grouped_欠損値', '間取り_grouped_１ＤＫ', '間取り_grouped_１Ｋ',
       '間取り_grouped_１ＬＤＫ', '間取り_grouped_１Ｒ', '間取り_grouped_２ＤＫ',
       '間取り_grouped_２Ｋ', '間取り_grouped_２ＬＤＫ', '間取り_grouped_２ＬＤＫ＋Ｓ',
       '間取り_grouped_３ＤＫ', '間取り_grouped_３ＬＤＫ', '間取り_grouped_４ＤＫ',
       '間取り_grouped_４ＬＤＫ','人口密度']]
# 目的変数のセット
Y = df['取引価格（総額）_log']

### 標準化 ###
# 説明変数の標準化（Zスコア）
X_standard =  preprocessing.scale(X) #各自入力

# 目的変数の標準化（Zスコア）
Y_standard = preprocessing.scale(Y) #各自入力

# y切片を追加設定 ※statsmodelの回帰モデルでは必須
X_const = sm.add_constant(X_standard)

### ③モデル構築 ###
# 重回帰モデルを作成
model = sm.OLS(Y_standard, X_const)  #インスタンス化（関数を使える状態にする） ※OLS=最小二乗法
results = model.fit()           #モデル構築（フィッティング）

In [31]:
X_const

array([[ 1.        , -0.19139362,  0.81935286, ..., -0.04923499,
        -0.31017845, -1.40787124],
       [ 1.        , -0.67776649, -1.04748129, ..., -0.04923499,
        -0.31017845, -1.40787124],
       [ 1.        ,  0.29497924,  0.25930262, ..., -0.04923499,
        -0.31017845, -1.40787124],
       ...,
       [ 1.        ,  3.21321642,  0.25930262, ..., -0.04923499,
        -0.31017845, -1.17082046],
       [ 1.        ,  0.94347639,  0.25930262, ..., -0.04923499,
        -0.31017845, -1.17082046],
       [ 1.        ,  0.45710353,  0.81935286, ..., -0.04923499,
         3.22395059, -1.17082046]])

In [44]:
### ④結果の出力 ###
results.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.627
Model:                            OLS   Adj. R-squared:                  0.627
Method:                 Least Squares   F-statistic:                 3.087e+04
Date:                Fri, 03 Oct 2025   Prob (F-statistic):               0.00
Time:                        15:46:01   Log-Likelihood:            -5.1094e+05
No. Observations:              551645   AIC:                         1.022e+06
Df Residuals:                  551614   BIC:                         1.022e+06
Df Model:                          30                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       5.687e-15      0.001   6.91e-12      1.000      -0.002       0.002
x1            -0.1281      0.001   -138.405      0.000      -0.130      -0.126
x2             0.3372      0.001    265.534      0.000       0.335       0.340
x3            -0.0603      0.001    -45.744      0.000      -0.063      -0.058
x4             0.1335      0.001     98.124      0.000       0.131       0.136
x5            -0.4365      0.001   -490.611      0.000      -0.438      -0.435
x6            -0.0063      0.001     -7.653      0.000      -0.008      -0.005
x7            -0.0042      0.001     -5.119      0.000      -0.006      -0.003
x8            -0.0004      0.001     -0.507      0.612      -0.002       0.001
x9            -0.0116      0.001    -14.060      0.000      -0.013      -0.010
x10           -0.0912      0.001   -110.156      0.000      -0.093      -0.090
x11           -0.0024      0.001     -2.936      0.003      -0.004      -0.001
x12           -0.0167      0.001    -20.336      0.000      -0.018      -0.015
x13           -0.0013      0.001     -1.534      0.125      -0.003       0.000
x14           -0.0002      0.001     -0.296      0.767      -0.002       0.001
x15            0.0677      0.001     79.655      0.000       0.066       0.069
x16           -0.0387      0.001    -32.140      0.000      -0.041      -0.036
x17           -0.0191      0.002    -10.202      0.000      -0.023      -0.015
x18           -0.0602      0.002    -32.199      0.000      -0.064      -0.056
x19           -0.1986      0.004    -54.026      0.000      -0.206      -0.191
x20            0.0143      0.002      6.678      0.000       0.010       0.019
x21           -0.0796      0.001    -60.262      0.000      -0.082      -0.077
x22           -0.0176      0.002     -9.306      0.000      -0.021      -0.014
x23           -0.0162      0.001    -17.405      0.000      -0.018      -0.014
x24            0.0570      0.003     16.371      0.000       0.050       0.064
x25            0.0099      0.001      8.895      0.000       0.008       0.012
x26           -0.0261      0.002    -17.155      0.000      -0.029      -0.023
x27            0.0541      0.005     11.244      0.000       0.045       0.064
x28           -0.0151      0.001    -15.869      0.000      -0.017      -0.013
x29            0.0335      0.003     11.755      0.000       0.028       0.039
x30            0.4053      0.001    452.514      0.000       0.404       0.407
==============================================================================
Omnibus:                   271345.946   Durbin-Watson:                   1.831
Prob(Omnibus):                  0.000   Jarque-Bera (JB):         15017908.005
Skew:                          -1.600   Prob(JB):                         0.00
Kurtosis:                      28.360   Cond. No.                         17.6
==

In [45]:
### VIF統計量を算出 ###

from statsmodels.stats.outliers_influence import variance_inflation_factor

# VIFを計算
vif = pd.DataFrame() #結果格納用のdataframeを準備
vif['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])] #VIFを計算しdataframeに格納
vif['変数名'] = X.columns #対応する変数名を格納

# VIFの計算結果を画面出力
display(vif)

,VIF,変数名
0,3.817784,最寄駅：距離（分）
1,12.282174,面積（㎡）
2,77.202200,建ぺい率（％）
3,13.387865,容積率（％）
4,3.794871,取引時点での築年数
5,1.000218,取引の事情等_その他事情有り
6,1.000133,取引の事情等_他の権利・負担付き
7,1.000020,取引の事情等_他の権利・負担付き、調停・競売等
8,1.000191,取引の事情等_瑕疵有りの可能性
9,1.037639,取引の事情等_調停・競売等


In [41]:
# ### グラフ（stem plot）による視覚化 ###

# # Figureサイズの指定
# plt.rcParams['figure.figsize'] = 10, 5
# # VIFの傾向をグラフ化
# plt.stem(vif.index.astype(str)+'.'+vif['変数名'], vif['VIF']) #x軸はインデックス番号と変数名を結合して表示
# # x軸の目盛文字を90度回転
# plt.xticks(rotation=90)
# # y軸ラベルを表示
# plt.ylabel('VIF')

In [46]:
### VIF=10以上の説明変数を抽出 ###

# VIF>=10に絞り込み
vif_over10 = vif[ vif['VIF']>=10 ] #各自入力（参考：vif[条件式]で条件抽出）
# 画面出力
display(vif_over10)

,VIF,変数名
1,12.282174,面積（㎡）
2,77.202200,建ぺい率（％）
3,13.387865,容積率（％）
18,10.106777,間取り_grouped_１Ｋ
26,26.543625,間取り_grouped_３ＬＤＫ


In [47]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

# 予測値の計算
Y_pred_standard = results.predict(X_const)

# MAE (平均絶対誤差) の計算
mae = mean_absolute_error(Y_standard, Y_pred_standard)
print(f"MAE (平均絶対誤差): {mae}")

# RMSE (二乗平均平方根誤差) の計算
rmse = np.sqrt(mean_squared_error(Y_standard, Y_pred_standard))
print(f"RMSE (二乗平均平方根誤差): {rmse}")

MAE (平均絶対誤差): 0.44821437167178485
RMSE (二乗平均平方根誤差): 0.6109623092779277
